In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 06 · MCP servers are OAuth 2.1 resource servers — practice

    **Primer section:** §7.1. Talk to the tickets MCP server as a raw OAuth client, then wire it into
    ADK with delegated, audience-bound tokens.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

import json

import httpx
import jwt  # display only
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from agentsec.agents import LocalStack, Step, build_support_agent
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.identity import DPoP, TokenIssuer, public_jwk
from agentsec.mcp import (
    SCOPE_READ,
    SCOPE_WRITE,
    TICKETS,
    ServerThread,
    build_server,
    delegated_token_minter,
    free_port,
    make_mcp_toolset,
)
from agentsec.runtime import run_turn, seed_session

issuer = TokenIssuer()      # the authorization server every party trusts
audit = AuditLog()

port = free_port()
url = f"http://127.0.0.1:{port}/mcp"          # the server's canonical URI = the RFC 8707 resource = token `aud`
srv = build_server(issuer, resource_url=url, audit=audit, require_dpop=False)
server = ServerThread(srv.app(require_dpop=False), port=port).start()
print("tickets MCP server listening at", url)

HEADERS = {
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
    "MCP-Protocol-Version": "2025-11-25",
}

def rpc(target_url: str, token: str | None = None, *, method: str = "tools/list", params: dict | None = None, headers: dict | None = None) -> httpx.Response:
    h = dict(HEADERS)
    if token:
        h["Authorization"] = f"Bearer {token}"
    h.update(headers or {})
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    return httpx.post(target_url, headers=h, json=body, timeout=10)

def call_tool(target_url: str, token: str | None, name: str, **arguments):
    return rpc(target_url, token, method="tools/call", params={"name": name, "arguments": arguments})

def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

## Exercise 1 — discover the resource server

Fetch the Protected Resource Metadata and check it names this server and our authorization server.
Then send an unauthenticated request and read the challenge.

In [ ]:
base = url.rsplit("/mcp", 1)[0]
prm = httpx.get(f"{base}/.well-known/____/mcp", timeout=5).json()
challenge = rpc(url)

assert prm["resource"].rstrip("/") == url
assert prm["authorization_servers"][0].rstrip("/") == issuer.issuer
assert SCOPE_READ in prm["scopes_supported"]
assert challenge.status_code == ____ and "resource_metadata=" in challenge.headers["www-authenticate"]
print(json.dumps(prm, indent=2))

## Exercise 2 — mint three tokens and predict the status codes

Mint (a) a token for another audience, (b) a token for this server with no scope, (c) a token for
this server with `tickets:read`. Fill in the expected HTTP status of `tools/list` for each.

In [ ]:
wrong_aud = issuer.mint(subject="u-ana", audience="https://other.example/mcp", scope=SCOPE_READ)
no_scope = issuer.mint(subject="u-ana", audience=____, scope="")
read_token = issuer.mint(subject="u-ana", audience=url, scope=____)

expected = {"wrong_aud": ____, "no_scope": ____, "read_token": ____}
actual = {"wrong_aud": rpc(url, wrong_aud).status_code, "no_scope": rpc(url, no_scope).status_code, "read_token": rpc(url, read_token).status_code}
assert actual == expected, actual
assert "insufficient_scope" in rpc(url, no_scope).headers["www-authenticate"]
print(actual)

## Exercise 3 — scope → tool

With the read token: list the tools and collect the names whose `readOnlyHint` is true; call
`get_ticket` for `T-1`; call `refund_ticket` and confirm it is a tool error mentioning
`insufficient_scope`.

In [ ]:
tools = rpc(url, read_token).json()["result"]["tools"]
read_only = sorted(t["name"] for t in tools if t["annotations"].get("____"))
got = call_tool(url, read_token, "get_ticket", ticket_id="T-1").json()["result"]
denied = call_tool(url, read_token, "refund_ticket", ticket_id="T-1", amount=10).json()["result"]

assert read_only == ["get_ticket", "list_tickets"]
assert got["structuredContent"]["ticket"]["owner"] == "u-ana"
assert denied["isError"] is ____ and "insufficient_scope" in denied["content"][0]["text"]
print("read-only tools:", read_only)

## Exercise 4 — DPoP: three presentations

Start a DPoP-required server, mint a token bound to a fresh key, and show: Bearer presentation →
401, `DPoP` + proof → 200, replayed proof → 401. Stop the server at the end.

In [ ]:
dpop_port = free_port()
dpop_url = f"http://127.0.0.1:{dpop_port}/mcp"
dpop_srv = build_server(issuer, resource_url=dpop_url, audit=AuditLog(), require_dpop=True)
dpop_server = ServerThread(dpop_srv.app(require_dpop=True), port=dpop_port).start()
try:
    key = DPoP.generate_key()
    bound = issuer.mint_dpop_bound_token(subject="u-ana", audience=dpop_url, dpop_public_jwk=____, scope=SCOPE_READ)
    proof = DPoP.proof(key, method="POST", url=dpop_url, access_token=____)

    as_bearer = rpc(dpop_url, bound).status_code
    with_proof = rpc(dpop_url, headers={"Authorization": f"____ {bound}", "DPoP": proof}).status_code
    replayed = rpc(dpop_url, headers={"Authorization": f"DPoP {bound}", "DPoP": proof}).status_code
    assert (as_bearer, with_proof, replayed) == (____, ____, ____), (as_bearer, with_proof, replayed)
    print("bearer:", as_bearer, "| proof:", with_proof, "| replay:", replayed)
finally:
    dpop_server.stop()

## Exercise 5 — the ADK toolset with delegated tokens, and the no-passthrough proof

Build the toolset with a delegated minter for this server's audience, run the scripted turn, and
assert row-level filtering, the forbidden cross-user read, the refund, and that the upstream
payments token has a **different** audience than the MCP server. Close the toolset and stop the server.

In [ ]:
stack = LocalStack.create(Settings(mcp_url=url, sts_issuer=issuer.issuer))
stack.issuer = issuer
minter = delegated_token_minter(issuer, agent=stack.agent_id, audience=____, scopes=[SCOPE_READ, SCOPE_WRITE])
toolset = make_mcp_toolset(url, token_minter=minter)
agent = build_support_agent(model=stack.llm, extra_tools=[toolset])
runner = Runner(app_name="mcp-lab", agent=agent, plugins=[stack.plugin], session_service=InMemorySessionService())
try:
    await seed_session(runner, user_id="u-ana", session_id="s1", user={"subject": "u-ana", "email": "ana@customer.example"}, scopes=[SCOPE_READ, SCOPE_WRITE])
    stack.script(
        Step.call("tickets_list_tickets"),
        Step.call("tickets_get_ticket", ticket_id="T-3"),
        Step.call("tickets_refund_ticket", ticket_id="T-2", amount=35.0),
        Step.say("ok"),
    )
    r = await run_turn(runner, user_id="u-ana", session_id="s1", message="tickets")
    by_name = {t["name"]: t["response"] for t in r.tool_responses}

    assert {t["id"] for t in by_name["tickets_list_tickets"]["structuredContent"]["tickets"]} == {"T-1", "T-2"}
    assert by_name["tickets_get_ticket"]["structuredContent"]["error"] == "forbidden"
    assert by_name["tickets_refund_ticket"]["structuredContent"]["ticket_id"] == "T-2" and TICKETS["T-2"]["status"] == "refunded"

    upstream = peek(srv.payments.calls[-1]["token"])
    assert upstream["aud"] == "https://payments.acme.example" and upstream["aud"] != ____
    assert upstream["sub"] == srv.payments.server_identity  # the MCP server's OWN identity, agent as actor
    assert upstream["act"]["sub"] == stack.agent_id.spiffe_id and upstream["on_behalf_of"] == "u-ana"
    print("upstream token audience:", upstream["aud"], "(inbound was for", url + ")")
finally:
    await toolset.____()
    server.stop()

**In one sentence:** "The MCP server validates that the token was issued for *it* — audience
and scope — publishes its metadata so clients can find the AS, annotates tools as read-only or
destructive, filters rows by the delegated subject, and gets its own token for anything upstream.
Through the gateway, DPoP makes the token useless without the key."